01_harmonize_province_names.py

Tugas Orang 5 — langkah pertama sebelum spatial join ke GeoJSON.

Masalah:
- cluster_membership.csv (output Orang 1) memakai nama ALL CAPS:
    "ACEH", "JAWA BARAT", "BANGKA BELITUNG", dst.
- GeoJSON 38 provinsi memakai nama Title Case resmi:
    "Aceh", "Jawa Barat", "Kepulauan Bangka Belitung", dst.
- Dua provinsi hasil pemekaran (Papua Tengah & Papua Pegunungan) ADA di GeoJSON
  tapi TIDAK muncul di data FCM (tidak ada observasi) → tampil abu-abu di peta.

Solusi:
- Membuat mapping ALL CAPS → GeoJSON name.
- Kasus khusus:
    "BANGKA BELITUNG"  → "Kepulauan Bangka Belitung"
    "DI YOGYAKARTA"    → "Daerah Istimewa Yogyakarta"
  (nama default Title Case tidak cukup untuk keduanya)

Output:
- data/interim/province_name_mapping.csv   (audit mapping)
- data/interim/fcm_membership_geojson.csv  (membership + nama GeoJSON, siap spatial join)

In [2]:
import json
from pathlib import Path

import pandas as pd

PROJECT_ROOT = Path().resolve().parent
GEOJSON_PATH    = PROJECT_ROOT / "data" / "external" / "indonesia_38_provinces.geojson"
MEMBERSHIP_PATH = PROJECT_ROOT / "outputs" / "model" / "cluster_membership.csv"
ANALYSIS_PATH   = PROJECT_ROOT / "outputs" / "analysis" / "province_membership_analysis.csv"
MAPPING_OUT     = PROJECT_ROOT / "data" / "interim" / "province_name_mapping.csv"
HARMONIZED_OUT  = PROJECT_ROOT / "data" / "interim" / "fcm_membership_geojson.csv"

# Kasus khusus: nama ALL CAPS tidak cukup di-titlecase karena
# GeoJSON pakai nama berbeda.
SPECIAL_MAPPING: dict[str, str] = {
    "BANGKA BELITUNG": "Kepulauan Bangka Belitung",
    "DI YOGYAKARTA":   "Daerah Istimewa Yogyakarta",
    "DKI JAKARTA":     "DKI Jakarta",  # DKI harus tetap huruf besar
}

# Provinsi di GeoJSON yang tidak punya data FCM (pemekaran 2022, belum ada survei)
NO_DATA_PROVINCES = {"Papua Tengah", "Papua Pegunungan"}

In [3]:
def load_geojson_names(path: Path) -> list[str]:
    with open(path, encoding="utf-8") as f:
        gj = json.load(f)
    return sorted(feat["properties"]["PROVINSI"] for feat in gj["features"])

In [4]:
def caps_to_geojson(caps_name: str, geo_set: set[str]) -> str | None:
    """Konversi nama ALL CAPS ke nama GeoJSON yang sesuai."""
    if caps_name in SPECIAL_MAPPING:
        return SPECIAL_MAPPING[caps_name]
    # Coba Title Case biasa
    title = caps_name.title()
    if title in geo_set:
        return title
    return None

In [5]:
def main():
    geo_names = load_geojson_names(GEOJSON_PATH)
    geo_set   = set(geo_names)
    print(f"Provinsi di GeoJSON : {len(geo_names)}")

    df_mem  = pd.read_csv(MEMBERSHIP_PATH)
    df_ana  = pd.read_csv(ANALYSIS_PATH)

    # Buat mapping table untuk audit
    rows = []
    for caps in df_mem["province_name"]:
        mapped = caps_to_geojson(caps, geo_set)
        if mapped is None:
            status = f"WARNING — tidak bisa dipetakan: {caps!r}"
        elif mapped not in geo_set:
            status = f"WARNING — {mapped!r} tidak ada di GeoJSON"
        else:
            method = "special" if caps in SPECIAL_MAPPING else "titlecase"
            status = f"OK ({method})"
        rows.append({"province_name_fcm": caps, "province_name_geojson": mapped, "status": status})

    mapping_df = pd.DataFrame(rows)
    MAPPING_OUT.parent.mkdir(parents=True, exist_ok=True)
    mapping_df.to_csv(MAPPING_OUT, index=False)

    warnings = mapping_df[mapping_df["status"].str.startswith("WARNING")]
    if not warnings.empty:
        print("\n[WARNING] Perlu diperiksa manual:")
        print(warnings.to_string(index=False))
    else:
        print(f"Semua {len(mapping_df)} provinsi FCM berhasil dipetakan ke GeoJSON.")

    # Buat dataset harmonisasi: membership + analisis, nama sudah GeoJSON
    df_merge = df_mem.copy()
    df_merge["province_name_geojson"] = df_merge["province_name"].apply(
        lambda x: caps_to_geojson(x, geo_set)
    )

    # Gabungkan dengan kolom membership_status & stunting_prevalence dari analisis Orang 4
    df_ana_slim = df_ana[["province_name", "membership_status", "stunting_prevalence_pct", "cluster_label"]].copy()
    df_merge = df_merge.merge(df_ana_slim, on="province_name", how="left")

    df_merge.to_csv(HARMONIZED_OUT, index=False)
    print(f"\nDataset harmonisasi disimpan → {HARMONIZED_OUT}")
    print(f"Kolom: {df_merge.columns.tolist()}")

    # Audit: provinsi GeoJSON tanpa data FCM
    mapped_geo = set(mapping_df["province_name_geojson"].dropna())
    missing = sorted(geo_set - mapped_geo)
    if missing:
        print(f"\n[INFO] Provinsi GeoJSON tanpa data FCM (akan abu-abu di peta): {missing}")

### Jalankan Pipeline

In [6]:
if __name__ == "__main__":
    main()

Provinsi di GeoJSON : 38
Semua 36 provinsi FCM berhasil dipetakan ke GeoJSON.

Dataset harmonisasi disimpan → C:\muti\SMT 6\PKK_PROJECT AKHIR\pkk-stunting-fcm-clustering\data\interim\fcm_membership_geojson.csv
Kolom: ['province_name', 'membership_cluster_1', 'membership_cluster_2', 'maximum_membership', 'second_highest_membership', 'membership_margin', 'crisp_cluster', 'province_name_geojson', 'membership_status', 'stunting_prevalence_pct', 'cluster_label']

[INFO] Provinsi GeoJSON tanpa data FCM (akan abu-abu di peta): ['Papua Pegunungan', 'Papua Tengah']
